# ResQRoute - crops + training on Kaggle GPU

Before running:
1. **Add Input** -> search `xview2-challenge-dataset-train-and-test` (by tunguz) -> Add.
2. **Add Input** -> your uploaded `resqroute-code` dataset (the repo's `scripts/` + `outputs/` folders).
3. **Settings** -> Accelerator: **GPU T4 x2** (or P100), **Internet: On** (ImageNet weights download).
4. Run All. At the end download `resqroute_results.zip` from the Output panel.

In [ ]:
import os, shutil, subprocess, pathlib
code_dir = next(p.parent.parent for p in pathlib.Path('/kaggle/input').rglob('scripts/common.py'))
work = pathlib.Path('/kaggle/working/resqroute')
for sub in ['scripts', 'outputs']:
    shutil.copytree(code_dir / sub, work / sub, dirs_exist_ok=True)
os.environ['RESQROUTE_ROOT'] = str(work)
os.chdir(work)
print('code from', code_dir); print(sorted(os.listdir(work / 'outputs')))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Step 3: cut 224x224 pre/post crops for every building in outputs/crop_manifest.csv
!pip -q install shapely
!python scripts/03_extract_crops.py --workers 4

In [ ]:
# Step 4: fine-tune the three classifiers (3-class); early stopping on val loss
for arch in ['mobilenet_v3_small', 'resnet18', 'efficientnet_b0']:
    !python scripts/04_train_model.py --arch {arch} --workers 4

In [ ]:
# Package crops + weights + training histories for download
!cd /kaggle/working/resqroute && zip -qr /kaggle/working/resqroute_results.zip data/crops models outputs
!ls -lh /kaggle/working/resqroute_results.zip models
os.chdir('/kaggle/working'); from IPython.display import FileLink; FileLink('resqroute_results.zip')